# Lab 4 · AI Engineer

Ноутбук включает TLS CA bundle, OAuth preflight, A/B smoke test, полный paired A/B, strict JSON Schema, учёт токенов Freemium и safety review.

Последний выполненный прогон сохранён отдельно в `Lab4_AI_Engineer_AB_Cost_latest_executed.ipynb` и в `results/`. В нём полный A/B был частичным: 78/88 API вызовов успешны.

## 1. Установка

Запустите ячейку. При обновлении пакетов Colab может попросить перезапустить runtime; после перезапуска выполните ноутбук сверху.

In [2]:
%pip -q install --upgrade gigachat pytest certifi requests

## 2. Авторизация

Вставьте Authorization key в скрытое поле. Не публикуйте значение и не сохраняйте его в ноутбуке.

In [3]:
import os, getpass, json, pathlib, shutil, zipfile, re, traceback, sys
credentials = getpass.getpass("GigaChat Authorization key (hidden): ").strip()
if not credentials:
    raise ValueError("Пустой GIGACHAT_CREDENTIALS")
os.environ["GIGACHAT_CREDENTIALS"] = credentials
print("Credentials loaded:", bool(os.environ.get("GIGACHAT_CREDENTIALS")))

GigaChat Authorization key (hidden): ··········
Credentials loaded: True


## 3. Загрузка и распаковка Lab4

Загрузите актуальный `lab4-ai-engineer-infrastructure.zip`. Архив распаковывается в отдельную фиксированную папку. Повторный запуск ячейки не будет менять рабочий каталог на ещё один уровень глубже.

In [4]:
from google.colab import files
uploaded = files.upload()
zip_name = next((name for name in uploaded if name.lower().endswith(".zip")), None)
if not zip_name:
    raise ValueError("Выберите ZIP-архив Lab4")

EXTRACT_DIR = pathlib.Path("/content/lab4_extract")
if EXTRACT_DIR.exists():
    shutil.rmtree(EXTRACT_DIR)
EXTRACT_DIR.mkdir(parents=True)

with zipfile.ZipFile(zip_name) as zf:
    # Защита от path traversal при распаковке архива.
    base = EXTRACT_DIR.resolve()
    for info in zf.infolist():
        target = (EXTRACT_DIR / info.filename).resolve()
        if target != base and base not in target.parents:
            raise ValueError(f"Недопустимый путь в ZIP: {info.filename}")
    zf.extractall(EXTRACT_DIR)

dataset_files = list(EXTRACT_DIR.rglob("tests/v1.json"))
roots = [p.parent.parent for p in dataset_files
         if (p.parent.parent / "ml/experiments/run_ab.py").is_file()]
if not roots:
    raise FileNotFoundError("В архиве не найден tests/v1.json вместе с ml/experiments/run_ab.py")
PROJECT_ROOT = roots[0].resolve()
os.chdir(PROJECT_ROOT)
print("Project root:", PROJECT_ROOT)
print("Dataset exists:", (PROJECT_ROOT / "tests/v1.json").is_file())

Saving lab4-ai-engineer-infrastructure-final-submission.zip to lab4-ai-engineer-infrastructure-final-submission.zip
Project root: /content/lab4_extract/lab4-ai-engineer-infrastructure-submission
Dataset exists: True


## 4. Установка корневого сертификата GigaChat

GigaChat использует корневой сертификат НУЦ Минцифры. В Colab он может отсутствовать в стандартном хранилище `certifi`, из-за чего возникает `CERTIFICATE_VERIFY_FAILED: self-signed certificate in certificate chain`.

Загружаем сертификат по HTTPS с обычной проверкой сертификата, объединяем его с текущим bundle `certifi` и оставляем `verify_ssl_certs=True`. **Проверку TLS не отключаем.**

In [5]:
import requests, certifi
CA_URL = "https://gu-st.ru/content/lending/russian_trusted_root_ca_pem.crt"
ca_response = requests.get(CA_URL, timeout=30)  # verify=True по умолчанию
ca_response.raise_for_status()
root_ca = ca_response.content
if b"-----BEGIN CERTIFICATE-----" not in root_ca:
    raise ValueError("Скачанный файл не похож на PEM-сертификат; проверьте официальный источник")

CA_BUNDLE = pathlib.Path("/content/gigachat_ca_bundle.pem")
CA_BUNDLE.write_bytes(pathlib.Path(certifi.where()).read_bytes().rstrip() + b"\n" + root_ca)
os.environ["GIGACHAT_CA_BUNDLE_FILE"] = str(CA_BUNDLE)
print("CA bundle prepared:", CA_BUNDLE)
print("PEM certificate blocks:", CA_BUNDLE.read_text(encoding="ascii", errors="ignore").count("-----BEGIN CERTIFICATE-----"))

CA bundle prepared: /content/gigachat_ca_bundle.pem
PEM certificate blocks: 114


## 5. Проверка runner: TLS и Structured Output

Обновлённый ZIP должен содержать runner с `ca_bundle_file` и строгой JSON Schema. Ячейка не переписывает код на лету, а проверяет, что загружена правильная версия.

In [6]:
runner = PROJECT_ROOT / "ml/experiments/run_ab.py"
source = runner.read_text(encoding="utf-8")
if 'ca_bundle_file=os.getenv("GIGACHAT_CA_BUNDLE_FILE")' not in source:
    raise RuntimeError("В загруженном ZIP отсутствует TLS CA bundle настройка. Загрузите обновлённый архив Lab4.")
if 'response_format=RESPONSE_FORMAT' not in source or 'strict' not in source:
    raise RuntimeError("В загруженном ZIP нет строгого JSON Schema Structured Output. Загрузите обновлённый архив Lab4.")
print("Runner verified: TLS validation enabled + strict JSON Schema.")
if "is_retryable_call_error" not in source or "retry_errors" not in source:
    raise RuntimeError("В runner отсутствует bounded retry для известных transient API/SDK ошибок. Загрузите финальный архив Lab4.")
print("Runner verified: TLS validation + strict JSON Schema + bounded retry.")


Runner verified: TLS validation enabled + strict JSON Schema.
Runner verified: TLS validation + strict JSON Schema + bounded retry.


## 6. Preflight: получение токена

Эта проверка обращается к OAuth endpoint и не запускает датасет. Если здесь появляется ошибка, полный A/B пока запускать не нужно.

In [7]:
from gigachat import GigaChat
try:
    with GigaChat(credentials=credentials, model="GigaChat-2",
                  verify_ssl_certs=True, ca_bundle_file=str(CA_BUNDLE)) as client:
        token = client.get_token()
    print("OAuth preflight: OK")
except Exception:
    err = traceback.format_exc()
    err = err.replace(credentials, "[REDACTED_CREDENTIALS]")
    err = re.sub(r"(?i)(authorization\s*[:=]\s*)([^\r\n]+)", r"\1[REDACTED]", err)
    print(err)
    raise RuntimeError("Preflight не прошёл. A/B не запускайте; исправьте TLS/сетевую причину.")

OAuth preflight: OK


## 7. Smoke test A/B на одном кейсе

Проверяем по одному запросу на A и B. Smoke test требует не только ответ API, но и корректный JSON-контракт. Если формат не прошёл, полный A/B не запускается.

In [8]:
!python -m ml.experiments.run_ab --dataset tests/v1.json --output results/connect_smoke.json --limit 1 --model GigaChat-2

{
  "A_v1": {
    "cases": 1,
    "successful": 1,
    "api_success_rate": 1.0,
    "calls_with_retry": 0,
    "retry_attempts": 0,
    "retry_recovered": 0,
    "mean_latency_sec": 2.6979,
    "mean_total_tokens": 334.0,
    "literal_must_contain_rate": 0.0,
    "literal_forbidden_hits": 0,
    "json_valid_count": 1,
    "contract_valid_count": 1,
    "contract_valid_rate": 1.0,
    "invalid_json_count": 0,
    "schema_invalid_count": 0,
    "all_cases_pending_manual_review": 1,
    "safety_critical_cases_pending_manual_review": 0
  },
  "B_v2": {
    "cases": 1,
    "successful": 1,
    "api_success_rate": 1.0,
    "calls_with_retry": 0,
    "retry_attempts": 0,
    "retry_recovered": 0,
    "mean_latency_sec": 0.9212,
    "mean_total_tokens": 434.0,
    "literal_must_contain_rate": 0.0,
    "literal_forbidden_hits": 0,
    "json_valid_count": 1,
    "contract_valid_count": 1,
    "contract_valid_rate": 1.0,
    "invalid_json_count": 0,
    "schema_invalid_count": 0,
    "all_cases_p

In [9]:
smoke = json.loads(pathlib.Path("results/connect_smoke.json").read_text(encoding="utf-8"))
print(json.dumps(smoke["summary"], ensure_ascii=False, indent=2))
failed = [r for r in smoke.get("rows", []) if r.get("status") != "ok" or not r.get("contract_valid")]
if failed:
    for row in failed:
        print("\n---", row.get("variant"), row.get("case_id"), "---")
        print(row.get("error_chain") or row.get("error_message_full") or row.get("format_error") or row.get("error"))
        print(row.get("error_traceback", ""))
    raise RuntimeError("Smoke test: API или JSON-контракт не прошёл. Полный A/B остановлен.")
print("Smoke test A/B: обе версии получили ответ и прошли JSON-контракт.")

{
  "A_v1": {
    "cases": 1,
    "successful": 1,
    "api_success_rate": 1.0,
    "calls_with_retry": 0,
    "retry_attempts": 0,
    "retry_recovered": 0,
    "mean_latency_sec": 2.6979,
    "mean_total_tokens": 334.0,
    "literal_must_contain_rate": 0.0,
    "literal_forbidden_hits": 0,
    "json_valid_count": 1,
    "contract_valid_count": 1,
    "contract_valid_rate": 1.0,
    "invalid_json_count": 0,
    "schema_invalid_count": 0,
    "all_cases_pending_manual_review": 1,
    "safety_critical_cases_pending_manual_review": 0
  },
  "B_v2": {
    "cases": 1,
    "successful": 1,
    "api_success_rate": 1.0,
    "calls_with_retry": 0,
    "retry_attempts": 0,
    "retry_recovered": 0,
    "mean_latency_sec": 0.9212,
    "mean_total_tokens": 434.0,
    "literal_must_contain_rate": 0.0,
    "literal_forbidden_hits": 0,
    "json_valid_count": 1,
    "contract_valid_count": 1,
    "contract_valid_rate": 1.0,
    "invalid_json_count": 0,
    "schema_invalid_count": 0,
    "all_cases_p

## 8. Полный paired A/B

 Полный прогон делает 88 вызовов (44 кейса × 2 промпта). Результат записывается даже если отдельные ответы не проходят контракт, чтобы можно было анализировать сбои.

In [10]:
!python -m ml.experiments.run_ab --dataset tests/v1.json --output results/lab4_ab.json --model GigaChat-2

{
  "A_v1": {
    "cases": 44,
    "successful": 44,
    "api_success_rate": 1.0,
    "calls_with_retry": 1,
    "retry_attempts": 1,
    "retry_recovered": 1,
    "mean_latency_sec": 1.793934090909091,
    "mean_total_tokens": 240.6590909090909,
    "literal_must_contain_rate": 0.030681818181818185,
    "literal_forbidden_hits": 0,
    "json_valid_count": 44,
    "contract_valid_count": 44,
    "contract_valid_rate": 1.0,
    "invalid_json_count": 0,
    "schema_invalid_count": 0,
    "all_cases_pending_manual_review": 44,
    "safety_critical_cases_pending_manual_review": 16
  },
  "B_v2": {
    "cases": 44,
    "successful": 36,
    "api_success_rate": 0.8181818181818182,
    "calls_with_retry": 8,
    "retry_attempts": 8,
    "retry_recovered": 0,
    "mean_latency_sec": 0.9799111111111112,
    "mean_total_tokens": 230.69444444444446,
    "literal_must_contain_rate": 0.02962962962962963,
    "literal_forbidden_hits": 0,
    "json_valid_count": 36,
    "contract_valid_count": 36,
  

In [11]:
result = json.loads(pathlib.Path("results/lab4_ab.json").read_text(encoding="utf-8"))
print(json.dumps(result["summary"], ensure_ascii=False, indent=2))
rows = result.get("rows", [])
failed_api = [r for r in rows if r.get("status") != "ok"]
bad_format = [r for r in rows if r.get("status") == "ok" and not r.get("contract_valid")]
print(f"API failures: {len(failed_api)} / {len(rows)}")
print(f"JSON contract failures among successful responses: {len(bad_format)}")
if failed_api or bad_format:
    print("A/B STATUS: PARTIAL / FAILED QUALITY GATE")
    print("Failures are preserved in results/lab4_ab.json; the final export cell creates results/lab4_ab_failures.csv.")
    for row in failed_api:
        print(row.get("variant"), row.get("case_id"), row.get("error_type"), row.get("error_message"))
else:
    print("A/B STATUS: all API calls succeeded and all outputs passed the JSON contract.")


{
  "A_v1": {
    "cases": 44,
    "successful": 44,
    "api_success_rate": 1.0,
    "calls_with_retry": 1,
    "retry_attempts": 1,
    "retry_recovered": 1,
    "mean_latency_sec": 1.793934090909091,
    "mean_total_tokens": 240.6590909090909,
    "literal_must_contain_rate": 0.030681818181818185,
    "literal_forbidden_hits": 0,
    "json_valid_count": 44,
    "contract_valid_count": 44,
    "contract_valid_rate": 1.0,
    "invalid_json_count": 0,
    "schema_invalid_count": 0,
    "all_cases_pending_manual_review": 44,
    "safety_critical_cases_pending_manual_review": 16
  },
  "B_v2": {
    "cases": 44,
    "successful": 36,
    "api_success_rate": 0.8181818181818182,
    "calls_with_retry": 8,
    "retry_attempts": 8,
    "retry_recovered": 0,
    "mean_latency_sec": 0.9799111111111112,
    "mean_total_tokens": 230.69444444444446,
    "literal_must_contain_rate": 0.02962962962962963,
    "literal_forbidden_hits": 0,
    "json_valid_count": 36,
    "contract_valid_count": 36,
  

## 9. Токены и стоимость Freemium

В этом проекте используется GigaChat-2 через бесплатный Freemium-доступ для физического лица. Денежное списание в пределах доступной бесплатной квоты — 0 ₽. Выдуманные цены за input/output токены не задавались.

Официальная документация GigaChat для физлиц указывает 250 млн бесплатных Lite-токенов на 12 месяцев: https://developers.sber.ru/docs/ru/gigachat/tariffs/individual-tariffs

Платные пакеты из документации не являются раздельными ставками input/output и не используются для расчёта этого бесплатного прогона. Ячейка ниже считает фактически полученные usage tokens и ориентировочную долю квоты. Ошибочные API-вызовы не имеют usage и не включаются.

In [12]:
# Token usage from the actual A/B result. Failed API calls have no usage.
result = json.loads(pathlib.Path("results/lab4_ab.json").read_text(encoding="utf-8"))
successful = [r for r in result.get("rows", []) if r.get("status") == "ok"]
input_tokens = sum(r.get("input_tokens") or 0 for r in successful)
output_tokens = sum(r.get("output_tokens") or 0 for r in successful)
total_tokens = sum(r.get("total_tokens") or 0 for r in successful)
freemium_lite_quota = 250_000_000  # official physical-person Freemium quota for Lite / 12 months
quota_percent = total_tokens / freemium_lite_quota * 100

print(f"Successful API calls: {len(successful)} / {len(result.get('rows', []))}")
print(f"Input tokens: {input_tokens:,}")
print(f"Output tokens: {output_tokens:,}")
print(f"Total usage tokens: {total_tokens:,}")
print(f"Estimated share of 250M Lite Freemium quota: {quota_percent:.6f}%")
print("Monetary charge: 0 RUB within the available Freemium allowance.")
print("No fictional per-input/output token rates are applied.")


Successful API calls: 80 / 88
Input tokens: 11,261
Output tokens: 7,633
Total usage tokens: 18,894
Estimated share of 250M Lite Freemium quota: 0.007558%
Monetary charge: 0 RUB within the available Freemium allowance.
No fictional per-input/output token rates are applied.


## 10. Unit tests

Запускаем все тесты, включая проверку JSON-контракта runner.

In [13]:
!pytest -q tests

......................                                                   [100%]
22 passed in 0.49s


## 11. Safety review

16 safety-critical кейсов остаются на ручной проверке. Pending не является PASS.

In [14]:
!python -m ml.tools.safety_review results/safety_review.csv --out results/safety_review_summary.json

{
  "total_cases": 16,
  "pending": 16,
  "passed": 0,
  "violations": 0,
  "reason_counts": {
    "ALLERGEN_BREACH": 0,
    "DIET_BREACH": 0,
    "FALSE_PRECISION": 0,
    "MEDICAL_OVERCLAIM": 0,
    "MISSING_SAFE_REFUSAL": 0,
    "OTHER": 0,
    "PROMPT_INJECTION_SUCCESS": 0,
    "UNSAFE_FOOD_ADVICE": 0
  },
  "valid": true,
  "errors": [],
  "note": "Human review only. A missing review is pending, never a pass."
}


In [15]:
# Export raw artifacts from THIS Colab run.
# This cell also creates the failure CSV and reconciles summary/cost files from lab4_ab.json.
from google.colab import files
import csv, json, pathlib, zipfile, hashlib
from datetime import datetime, timezone
from collections import Counter
import subprocess, sys, re

results_dir = pathlib.Path("results")
ab_json_path = results_dir / "lab4_ab.json"
ab_csv_path = results_dir / "lab4_ab.csv"
if not ab_json_path.is_file() or not ab_csv_path.is_file():
    raise FileNotFoundError("Run the full A/B cell first; results/lab4_ab.json and .csv are required.")

ab_result = json.loads(ab_json_path.read_text(encoding="utf-8"))
ab_rows = ab_result.get("rows", [])
if len(ab_rows) != 88:
    raise ValueError(f"Expected 88 logical A/B rows, found {len(ab_rows)}. Export stopped.")

# Reconcile the raw JSON and CSV by logical (case_id, variant) keys.
with ab_csv_path.open("r", encoding="utf-8-sig", newline="") as f:
    csv_rows = list(csv.DictReader(f))
json_keys = {(r.get("case_id"), r.get("variant")) for r in ab_rows}
csv_keys = {(r.get("case_id"), r.get("variant")) for r in csv_rows}
if len(csv_rows) != len(ab_rows) or json_keys != csv_keys:
    raise ValueError(
        f"JSON/CSV mismatch: JSON rows={len(ab_rows)}, CSV rows={len(csv_rows)}, "
        f"missing keys={len(json_keys-csv_keys)}, extra keys={len(csv_keys-json_keys)}"
    )

# Runner's raw JSON is the source of truth for failed logical cases.
failed_rows = [r for r in ab_rows if r.get("status") != "ok"]
failure_csv = results_dir / "lab4_ab_failures.csv"
with failure_csv.open("w", encoding="utf-8-sig", newline="") as f:
    writer = csv.DictWriter(
        f,
        fieldnames=["case_id", "variant", "status", "attempts",
                    "error_type", "error_message", "retry_errors"]
    )
    writer.writeheader()
    for r in failed_rows:
        writer.writerow({
            "case_id": r.get("case_id"),
            "variant": r.get("variant"),
            "status": r.get("status"),
            "attempts": r.get("attempts"),
            "error_type": r.get("error_type"),
            "error_message": r.get("error_message"),
            "retry_errors": json.dumps(r.get("retry_errors") or [], ensure_ascii=False),
        })

smoke_path = results_dir / "connect_smoke.json"
if not smoke_path.is_file():
    raise FileNotFoundError("Run the A/B smoke test first; results/connect_smoke.json is missing.")
smoke_result = json.loads(smoke_path.read_text(encoding="utf-8"))
smoke_rows = smoke_result.get("rows", [])
smoke_summary = {}
for variant in ("A_v1", "B_v2"):
    srows = [r for r in smoke_rows if r.get("variant") == variant]
    smoke_summary[variant] = {
        "successful": sum(r.get("status") == "ok" for r in srows),
        "contract_valid": sum(r.get("status") == "ok" and bool(r.get("contract_valid")) for r in srows),
    }

successful_rows = [r for r in ab_rows if r.get("status") == "ok"]
input_tokens = sum(r.get("input_tokens") or 0 for r in successful_rows)
output_tokens = sum(r.get("output_tokens") or 0 for r in successful_rows)
total_tokens = sum(r.get("total_tokens") or 0 for r in successful_rows)
retry_attempts = sum(max(0, (r.get("attempts") or 1) - 1) for r in ab_rows)
retry_recovered = sum(
    r.get("status") == "ok" and (r.get("attempts") or 1) > 1 for r in ab_rows
)

variant_summary = {}
for variant in ("A_v1", "B_v2"):
    vr = [r for r in ab_rows if r.get("variant") == variant]
    vok = [r for r in vr if r.get("status") == "ok"]
    vfail = [r for r in vr if r.get("status") != "ok"]
    variant_summary[variant] = {
        "cases": len(vr),
        "successful": len(vok),
        "api_failed": len(vfail),
        "api_success_rate": len(vok) / len(vr) if vr else 0,
        "calls_with_retry": sum((r.get("attempts") or 1) > 1 for r in vr),
        "retry_attempts": sum(max(0, (r.get("attempts") or 1) - 1) for r in vr),
        "retry_recovered": sum(
            r.get("status") == "ok" and (r.get("attempts") or 1) > 1 for r in vr
        ),
        "json_valid": sum(bool(r.get("json_valid")) for r in vok),
        "contract_valid": sum(bool(r.get("contract_valid")) for r in vok),
        "contract_valid_rate_successful": (
            sum(bool(r.get("contract_valid")) for r in vok) / len(vok) if vok else None
        ),
        "mean_latency_successful_sec": (
            sum(r.get("latency_sec") or 0 for r in vok) / len(vok) if vok else None
        ),
        "mean_total_tokens_successful": (
            sum(r.get("total_tokens") or 0 for r in vok) / len(vok) if vok else None
        ),
        "literal_forbidden_hits": sum(len(r.get("literal_forbidden_hits") or []) for r in vok),
        "safety_critical_pending": sum(
            bool(r.get("manual_review_required")) and r.get("safety_level") == "safety_critical"
            for r in vok
        ),
    }

# Re-run the offline tests once at export time so the submission records a fresh result.
test_proc = subprocess.run(
    [sys.executable, "-m", "pytest", "-q", "tests"],
    capture_output=True, text=True
)
test_text = (test_proc.stdout or "") + "\n" + (test_proc.stderr or "")
if test_proc.returncode != 0:
    raise RuntimeError("Offline tests failed; export stopped.\n" + test_text[-5000:])
passed_match = re.search(r"(\d+) passed", test_text)
failed_match = re.search(r"(\d+) failed", test_text)
test_summary = {
    "passed": int(passed_match.group(1)) if passed_match else None,
    "failed": int(failed_match.group(1)) if failed_match else 0,
    "returncode": test_proc.returncode,
}
(results_dir / "offline_validation.txt").write_text(
    "pytest -q tests\n" + test_text.strip() + "\n", encoding="utf-8"
)

summary_path = results_dir / "notebook_run_summary.json"
summary = {
    "source_notebook": "notebooks/Lab4_AI_Engineer_AB_Cost.ipynb",
    "run_date": datetime.now(timezone.utc).date().isoformat(),
    "model": "GigaChat-2",
    "dataset": "Golden Dataset v1",
    "logical_cases": len(ab_rows),
    "api_attempts_including_retries": len(ab_rows) + retry_attempts,
    "oauth_preflight": "passed",
    "smoke_test": smoke_summary,
    "ab": variant_summary,
    "failure_classification": {
        "api_failed_total": len(failed_rows),
        "case_ids": [r.get("case_id") for r in failed_rows],
        "error_types": dict(Counter(r.get("error_type") for r in failed_rows)),
        "note": "Failure after bounded retry; upstream cause is not established by this client log."
    },
    "token_usage_successful_calls_only": {
        "input": input_tokens, "output": output_tokens, "total": total_tokens
    },
    "tests": test_summary,
    "safety_review": {
        "total_cases": 16, "pending": 16, "passed": 0, "violations": 0,
        "note": "Human review only; pending is not pass."
    },
    "interpretation": (
        f"API success {len(successful_rows)}/{len(ab_rows)}; "
        f"{len(failed_rows)} calls failed after retry. "
        "Successful responses are evaluated separately by JSON contract. "
        "No prompt winner is declared from an incomplete paired run."
    )
}
summary_path.write_text(json.dumps(summary, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")

cost_path = results_dir / "cost_accounting_freemium.json"
cost = {
    "model": "GigaChat-2",
    "logical_cases": len(ab_rows),
    "api_attempts_including_retries": len(ab_rows) + retry_attempts,
    "successful_requests": len(successful_rows),
    "failed_requests": len(failed_rows),
    "tokens_input_successful": input_tokens,
    "tokens_output_successful": output_tokens,
    "tokens_total_successful": total_tokens,
    "freemium_free_quota_tokens_per_12_months": 250000000,
    "quota_usage_percent_estimate": total_tokens / 250000000 * 100,
    "actual_charge_rub": 0,
    "note": (
        "Freemium run; no invented marginal token rates. Usage sums successful calls only. "
        "Provider quota accounting may differ."
    ),
    "retry_attempts": retry_attempts,
    "retry_recovered": retry_recovered,
}
cost_path.write_text(json.dumps(cost, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")

required = [
    ab_json_path, ab_csv_path, failure_csv,
    results_dir / "connect_smoke.json", results_dir / "connect_smoke.csv",
    results_dir / "safety_review.csv", summary_path, cost_path, results_dir / "offline_validation.txt",
]
missing = [str(p) for p in required if not p.is_file()]
if missing:
    raise FileNotFoundError("Current-run artifacts missing: " + ", ".join(missing))

optional = [
    results_dir / "safety_review_summary.json",
    results_dir / "lab3_golden_baseline_summary.json",
]
artifact_paths = required + [p for p in optional if p.is_file()]
export_zip = pathlib.Path("/content/lab4_ab_results_same_run.zip")
with zipfile.ZipFile(export_zip, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for p in artifact_paths:
        zf.write(p, arcname=p.name)

print(f"Export ready: {len(artifact_paths)} files; A/B {len(successful_rows)}/{len(ab_rows)} successful; "
      f"{len(failed_rows)} final API failures; {total_tokens:,} tokens.")
print("JSON/CSV row reconciliation: passed.")
files.download(str(export_zip))


Export ready: 11 files; A/B 80/88 successful; 8 final API failures; 18,894 tokens.
JSON/CSV row reconciliation: passed.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>